# Clase 2 — Datos y datasets en PyTorch

Curso **"Deep Learning con Python y PyTorch desde cero"** — DIH DATAlife (Hub de Innovación Digital Europeo), a partir de la certificación EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

**Duración estimada de la clase:** 100 min.

## Objetivos de aprendizaje de esta clase

- Explicar por qué los datos son la base del entrenamiento y qué formatos admite PyTorch
- Usar la clase `Dataset` para representar un conjunto de datos
- Usar `DataLoader` para servir los datos en lotes (*batches*) durante el entrenamiento
- Cargar un dataset de referencia (MNIST) con `torchvision` y aplicar transformaciones básicas

Este notebook acompaña al capítulo 02 del manual del curso. Ejecuta cada celda en orden con **Mayús + Intro**. Puedes ejecutarlo tal cual en Google Colab: solo necesita `torch`, `torchvision` y `matplotlib`, y la primera vez que cargues MNIST se descargará automáticamente.

## Parte 1 — Por qué importa este tema

En la clase anterior construiste una red neuronal mínima con `nn.Module`. Esa red no aprende nada por sí sola: necesita datos con los que entrenar. En Deep Learning (DL, aprendizaje profundo), preparar, cargar e iterar sobre los datos suele consumir más tiempo que diseñar la arquitectura de la red — se calcula que puede llegar a representar el 90% del trabajo real de un proyecto de Inteligencia Artificial (IA).

Antes de entrenar nada, PyTorch necesita responder a tres preguntas sobre tus datos:

- ¿En qué formato están y cómo se representan como tensores?
- ¿Cómo se dividen en un conjunto de entrenamiento y otro de prueba (*test*)?
- ¿Cómo se le entregan al modelo durante el entrenamiento sin agotar la memoria del ordenador?

PyTorch resuelve esto con dos piezas que trabajan juntas: `Dataset` y `DataLoader`.

## Parte 2 — La clase Dataset

Un `Dataset` es un contrato muy simple: cualquier objeto que lo cumple sabe responder a dos preguntas:

- `__len__`: ¿cuántos elementos tiene en total?
- `__getitem__`: ¿cómo es el elemento número *i*?

Da igual si por debajo hay una lista de números o cien mil imágenes en disco: mientras el objeto responda a esas dos preguntas, PyTorch lo trata como un dataset. Puedes construir tu propio `Dataset` heredando de `torch.utils.data.Dataset`, o usar uno de los que ya vienen listos en `torchvision`. Hoy empiezas con uno ya hecho — MNIST — para centrarte en el mecanismo antes que en construir tus propios datos.

In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

print("Versión de torch:", torch.__version__)
print("Versión de torchvision:", torchvision.__version__)

## Parte 3 — Qué aporta torchvision

`torchvision` es el módulo de PyTorch centrado en visión por computador. Aporta dos cosas que vas a usar ahora mismo:

- **Datasets de referencia** ya empaquetados como clases `Dataset`: MNIST (dígitos escritos a mano), CIFAR-10 (imágenes en color de diez categorías) e ImageNet (millones de imágenes con miles de categorías), entre otros
- **Transformaciones**, en `torchvision.transforms`, que convierten los datos crudos en el formato que espera un modelo. Aquí usarás la más básica: `transforms.ToTensor()`, que convierte una imagen en un tensor de PyTorch

Vamos a cargar MNIST: la parte de entrenamiento y la de prueba (*test*), por separado. Si es la primera vez que ejecutas esta celda, `download=True` descargará los datos a la carpeta `./datos`.

In [ ]:
# Transformación básica: convierte cada imagen (en formato PIL) en un tensor de PyTorch
transformacion = transforms.ToTensor()

# Dataset de entrenamiento: se usa para ajustar los parámetros del modelo
trainset = datasets.MNIST(
    root="./datos",
    train=True,
    download=True,
    transform=transformacion,
)

# Dataset de prueba (test): datos que el modelo nunca ve durante el entrenamiento,
# reservados para comprobar si generaliza bien
testset = datasets.MNIST(
    root="./datos",
    train=False,
    download=True,
    transform=transformacion,
)

print("Ejemplos de entrenamiento:", len(trainset))
print("Ejemplos de test:", len(testset))

## Parte 4 — DataLoader y entrenamiento por lotes

Tener el dataset organizado no basta: durante el entrenamiento necesitas servírselo al modelo en porciones manejables, llamadas **lotes** (*batches*). `torch.utils.data.DataLoader` agrupa el dataset en lotes de un tamaño que tú decides (`batch_size`) y te permite iterar sobre ellos con un simple bucle `for`.

Entrenar por lotes tiene varias ventajas frente a usar todo el dataset de golpe:

- Un dataset grande puede no entrar entero en memoria; un lote sí
- El modelo actualiza sus parámetros varias veces por cada paso completo (*época*) por el dataset, en lugar de una sola vez
- Un tamaño de lote habitual está entre ocho y 64 ejemplos

El parámetro `shuffle=True` mezcla el orden de los ejemplos antes de cada época. Esto importa especialmente en MNIST: si el modelo viera primero todos los "ceros", después todos los "unos", y así sucesivamente, podría aprender atajos ligados a esa posición en vez de a las características reales de cada dígito.

In [ ]:
# DataLoader de entrenamiento: lotes de 10 ejemplos, mezclados en cada época
trainloader = DataLoader(trainset, batch_size=10, shuffle=True)

# DataLoader de test: mismo tamaño de lote, también mezclado (aquí el orden es menos crítico)
testloader = DataLoader(testset, batch_size=10, shuffle=True)

print("Número de lotes de entrenamiento:", len(trainloader))
print("Número de lotes de test:", len(testloader))

In [ ]:
# Iteramos un único lote para inspeccionarlo
for datos in trainloader:
    X, y = datos  # X: imágenes del lote, y: etiquetas (el dígito de cada imagen)
    break  # con el primer lote nos basta, no queremos recorrer todo el dataset

print("Forma del lote de imágenes:", X.shape)   # [batch_size, canales, alto, ancho]
print("Forma del lote de etiquetas:", y.shape)
print("Etiquetas del lote:", y)

## Parte 5 — Visualizar un lote

Cada imagen del lote tiene forma `(1, 28, 28)`: un canal de color (escala de grises) y 28x28 píxeles. Para dibujarla con `matplotlib` necesitas quitar la dimensión del canal con `.view(28, 28)` o `.squeeze()`. Vamos a mostrar las diez imágenes del lote junto a su etiqueta.

In [ ]:
import matplotlib.pyplot as plt

figura, ejes = plt.subplots(1, 10, figsize=(15, 2))

for i in range(10):
    imagen = X[i][0].view(28, 28)  # quitamos la dimensión del canal: (1, 28, 28) -> (28, 28)
    ejes[i].imshow(imagen, cmap="gray")
    ejes[i].set_title(str(y[i].item()))
    ejes[i].axis("off")

plt.tight_layout()
plt.show()

## Parte 6 — ¿Está equilibrado el dataset?

Un dataset está **equilibrado** cuando cada categoría (aquí, cada dígito del cero al nueve) tiene un número parecido de ejemplos. Si una categoría domina el dataset, el modelo puede aprender el atajo de predecir siempre esa opción para minimizar el error, en lugar de aprender a distinguir de verdad entre dígitos.

Vamos a contar cuántos ejemplos hay de cada dígito en el conjunto de entrenamiento, recorriendo el dataset con el `DataLoader`.

In [ ]:
from collections import Counter

contador = Counter()

for _, etiquetas in trainloader:
    contador.update(etiquetas.tolist())

total = sum(contador.values())
for digito in sorted(contador):
    porcentaje = 100 * contador[digito] / total
    print(f"Dígito {digito}: {contador[digito]} ejemplos ({porcentaje:.1f}%)")

## Resumen de la clase

- Los datos son la base del entrenamiento: prepararlos y organizarlos suele llevar más tiempo que diseñar la red
- Un `Dataset` es cualquier objeto que sabe cuántos elementos tiene (`__len__`) y cómo devolver el ejemplo número *i* (`__getitem__`)
- `torchvision` ofrece datasets de referencia (MNIST, CIFAR-10, ImageNet) y transformaciones (`torchvision.transforms`) listas para usar
- `DataLoader` sirve los datos en lotes (`batch_size`), permite mezclarlos (`shuffle=True`) y evita cargar todo el dataset en memoria de golpe
- Comprobar si un dataset está equilibrado —contando ejemplos por clase— ayuda a detectar sesgos antes de entrenar

En el notebook de práctica vas a construir tu propio `Dataset` personalizado y a calcular cuántos lotes resultan de dividir MNIST con distintos valores de `batch_size`.